# FINPLE canonical CSV monthly one-click build

Edit only `AS_OF_DATE`, then run all cells in Colab. Outputs stay in Google Drive; this notebook never replaces the runtime CSV or performs a Production operation.

In [ ]:
# Edit only this value for the next monthly run.
AS_OF_DATE = "2026-07-28"

REPO_REF = "main"
REPO_URL = "https://github.com/vip930sw/FINPLE.git"
DRIVE_ROOT = "/content/drive/MyDrive/FINPLE"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/FINPLE")
if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

CHECKOUT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"Checked out {REPO_REF}: {CHECKOUT_SHA}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "yfinance"],
    check=True,
)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))


In [ ]:
DRIVE_ROOT_PATH = Path(DRIVE_ROOT).resolve()
RUN_DIR = (DRIVE_ROOT_PATH / f"canonical_run_{AS_OF_DATE}").resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CANONICAL_PATH = (
    REPO_DIR / "src/data/tickers/finple_app_candidates_v2.csv"
).resolve()
BENCHMARK_POLICY_PATH = (
    REPO_DIR / "tools/canonical_csv/benchmark_policy.example.csv"
).resolve()
UNIVERSE_PATH = (DRIVE_ROOT_PATH / "editable-universe.csv").resolve()
CACHE_DIR = (DRIVE_ROOT_PATH / "canonical_csv_cache").resolve()
OUTPUT_CANDIDATE_PATH = (RUN_DIR / "finple_app_candidates_v2.candidate.csv").resolve()
VALIDATION_PATH = (RUN_DIR / "validation.json").resolve()
FAILED_PATH = (RUN_DIR / "failed.csv").resolve()
SUMMARY_PATH = (RUN_DIR / "summary.json").resolve()
CHECKPOINT_PATH = (RUN_DIR / "checkpoint.json").resolve()
UNIVERSE_REPORT_PATH = (RUN_DIR / "universe-update.json").resolve()
OPERATOR_SUMMARY_PATH = (RUN_DIR / "operator-summary.json").resolve()

for path in (
    UNIVERSE_PATH,
    CACHE_DIR,
    OUTPUT_CANDIDATE_PATH,
    VALIDATION_PATH,
    FAILED_PATH,
    SUMMARY_PATH,
    CHECKPOINT_PATH,
    OPERATOR_SUMMARY_PATH,
):
    if not path.is_absolute():
        raise ValueError(f"Drive output path must be absolute: {path}")


In [ ]:
import csv

if UNIVERSE_PATH.exists():
    UNIVERSE_ACTION = "monthly_update"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tools.canonical_csv.update_universe",
            "--existing-universe",
            str(UNIVERSE_PATH),
            "--source-canonical",
            str(SOURCE_CANONICAL_PATH),
            "--benchmark-policy",
            str(BENCHMARK_POLICY_PATH),
            "--output",
            str(UNIVERSE_PATH),
            "--diff-report",
            str(UNIVERSE_REPORT_PATH),
        ],
        check=True,
    )
else:
    UNIVERSE_ACTION = "bootstrap"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "tools.canonical_csv.bootstrap_universe",
            "--source-canonical",
            str(SOURCE_CANONICAL_PATH),
            "--benchmark-policy",
            str(BENCHMARK_POLICY_PATH),
            "--output",
            str(UNIVERSE_PATH),
        ],
        check=True,
    )

with UNIVERSE_PATH.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    universe_headers = list(reader.fieldnames or ())
    universe_rows = list(reader)

UNRESOLVED_EXCLUDED_COUNT = 0
for row in universe_rows:
    if not str(row.get("marketDataProviderSymbol") or "").strip():
        row["includeInSimulator"] = "false"
        row["reasonCode"] = "market_data_provider_symbol_unresolved"
        row["reasonMessage"] = "marketDataProviderSymbol is unresolved"
        UNRESOLVED_EXCLUDED_COUNT += 1

temporary_universe = UNIVERSE_PATH.with_suffix(".tmp")
with temporary_universe.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=universe_headers, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(universe_rows)
os.replace(temporary_universe, UNIVERSE_PATH)
print(f"Universe: {UNIVERSE_ACTION}; unresolved excluded: {UNRESOLVED_EXCLUDED_COUNT}")


In [ ]:
from tools.canonical_csv.build import build_canonical_candidate
from tools.canonical_csv.cache import PersistentCachedMarketDataProvider
from tools.canonical_csv.config import PipelineConfig
from tools.canonical_csv.market_data import YFinanceMarketDataProvider

CHUNK_SIZE = 100
RESUME = True
RETRY_COUNT = 3
RETRY_BACKOFF_SECONDS = 5
ROLLING_CAGR_WINDOW_YEARS = (10, 7, 5, 3, 1)
MIN_ROLLING_WINDOWS = 6

config = PipelineConfig(
    source_canonical_path=SOURCE_CANONICAL_PATH,
    universe_path=UNIVERSE_PATH,
    output_candidate_path=OUTPUT_CANDIDATE_PATH,
    validation_report_path=VALIDATION_PATH,
    failed_assets_path=FAILED_PATH,
    run_summary_path=SUMMARY_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    as_of_date=__import__("datetime").date.fromisoformat(AS_OF_DATE),
    cache_dir=CACHE_DIR,
    chunk_size=CHUNK_SIZE,
    resume=RESUME,
    retry_count=RETRY_COUNT,
    retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
    rolling_cagr_window_years=ROLLING_CAGR_WINDOW_YEARS,
    min_rolling_windows=MIN_ROLLING_WINDOWS,
    write_non_publishable_candidate=True,
)
provider = PersistentCachedMarketDataProvider(
    YFinanceMarketDataProvider(),
    config.cache_dir,
    retry_count=config.retry_count,
    retry_backoff_seconds=config.retry_backoff_seconds,
)

PIPELINE_ERROR = ""
try:
    result = build_canonical_candidate(config, provider)
except Exception as error:
    PIPELINE_ERROR = f"{type(error).__name__}: {error}"
    print(f"Pipeline finished with a reviewable error: {PIPELINE_ERROR}")


In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

validation = read_json(VALIDATION_PATH)
run_summary = read_json(SUMMARY_PATH)
operator_summary = {
    "asOfDate": AS_OF_DATE,
    "repoRef": REPO_REF,
    "checkoutSha": CHECKOUT_SHA,
    "universeAction": UNIVERSE_ACTION,
    "unresolvedProviderAssetsExcluded": UNRESOLVED_EXCLUDED_COUNT,
    "structuralValid": validation.get("structuralValid"),
    "publishable": validation.get("publishable"),
    "pipelineError": PIPELINE_ERROR,
    "runtimeCsvReplacementPerformed": False,
    "outputRowCount": run_summary.get("outputRowCount"),
    "simulatorReadyRowCount": validation.get("simulatorReadyRowCount"),
}
OPERATOR_SUMMARY_PATH.write_text(
    json.dumps(operator_summary, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

ZIP_PATH = RUN_DIR / f"finple-canonical-{AS_OF_DATE}-{CHECKOUT_SHA[:12]}.zip"
artifacts = (
    OUTPUT_CANDIDATE_PATH,
    VALIDATION_PATH,
    FAILED_PATH,
    SUMMARY_PATH,
    CHECKPOINT_PATH,
    UNIVERSE_REPORT_PATH,
    OPERATOR_SUMMARY_PATH,
)
with ZipFile(ZIP_PATH, "w", ZIP_DEFLATED) as archive:
    archive.writestr("checkout-sha.txt", CHECKOUT_SHA + "\n")
    for artifact in artifacts:
        if artifact.exists():
            archive.write(artifact, artifact.name)

print(json.dumps(operator_summary, ensure_ascii=False, indent=2, sort_keys=True))
print(f"Checkout SHA: {CHECKOUT_SHA}")
print(f"Result ZIP: {ZIP_PATH}")

from google.colab import files
files.download(str(ZIP_PATH))
